# Summary Statistics on Electricians by MSA
This should be interesting to get an overview of how many electricians there are in Hawaii as well as other areas.

In [2]:
import pandas as pd
import numpy as np
from plotnine import *
import seaborn as sns
import statsmodels.api as sm
from great_tables import GT
import requests
from io import StringIO

In [3]:
# obtaining county fips and other info from simplemaps 

county_fips = pd.read_csv("raw-data/simplemaps_uscounties_basicv1.92/uscounties.csv")
print(county_fips.head()) #perfect

        county county_ascii         county_full  county_fips state_id  \
0  Los Angeles  Los Angeles  Los Angeles County         6037       CA   
1         Cook         Cook         Cook County        17031       IL   
2       Harris       Harris       Harris County        48201       TX   
3     Maricopa     Maricopa     Maricopa County         4013       AZ   
4    San Diego    San Diego    San Diego County         6073       CA   

   state_name      lat       lng  population  
0  California  34.3219 -118.2247     9808667  
1    Illinois  41.8401  -87.8168     5182090  
2       Texas  29.8578  -95.3938     4838303  
3     Arizona  33.3490 -112.4915     4559748  
4  California  33.0343 -116.7350     3288774  


# Strategy of Reading in Data

I read in every county, and within each county, I filter for only the electricians or whatever NAICS 6-digit code I have. All of this is then assembled into one dataframe, that is exported. 

The issue is, what is a county and what is a state? We'll figure this out.

In [16]:
#helper to determine the URL

def qcew_reader(year, quarter, data_type, code):
    url = f"https://data.bls.gov/cew/data/api/{year}/{quarter}/{data_type}/{code}.csv"
    r = requests.get(url)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text))
    return df

# we assume default is annualized, 
# we want a panel from year to year, 
# and then we want industry data of a specific 6-digit NAICS
def qcew_series(years, data_type, code):
    final = pd.DataFrame()
    for y in years:
        df = qcew_reader(y, "a", data_type, code)
        # print(y, df.shape) #check
        df["year"] = y # have to make a year var
        final = pd.concat([final, df], ignore_index= True)
    return final

def qcew_downloader(year, quarter, data_type, code):
    url = f"https://data.bls.gov/cew/data/api/{year}/{quarter}/{data_type}/{code}.csv"
    filename = "electricians-qcew.bin"

    with requests.get(url, stream = True) as r:
        r.raise_for_status()
        with open(filename, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):  # 1 MB
                if not chunk:
                    continue
                f.write(chunk)
                f.flush()  # make sure OS flushes to disk


## programmatic way to read in county and other fips, but already have simplemaps dictionary
# county_url = "https://data.bls.gov/cew/doc/titles/area/area_titles.csv"
# r = requests.get(county_url)
# r.raise_for_status()
# county_fips = pd.read_csv(StringIO(r.text))
# print(county_fips.tail(), '\n', county_fips.shape)


In [5]:
df = qcew_reader(2020, "a", "industry", "3361") #a is annual average
df.head()
print(df.shape)

(502, 38)


In [6]:
years = range(2018, 2023,1)
test = qcew_series(years, "industry", "3361")
print(type(test),'\n', test.head())

2018 (488, 38)
2019 (475, 38)
2020 (502, 38)
2021 (520, 38)
2022 (525, 38)
<class 'pandas.DataFrame'> 
   area_fips  own_code  industry_code  agglvl_code  size_code  year qtr  \
0     01000         5           3361           56          0  2018   A   
1     01015         5           3361           76          0  2018   A   
2     01019         5           3361           76          0  2018   A   
3     01073         5           3361           76          0  2018   A   
4     01083         5           3361           76          0  2018   A   

  disclosure_code  annual_avg_estabs  annual_avg_emplvl  ...  \
0             NaN                 21              13500  ...   
1               N                  1                  0  ...   
2               N                  1                  0  ...   
3             NaN                  4                288  ...   
4               N                  1                  0  ...   

   oty_total_annual_wages_chg  oty_total_annual_wages_pct_chg  \
0

In [7]:
years = range(2015, 2018,1)
df = qcew_series(years, "area", "15000")
# df = df[df["industry_code"] == "238210"] # returns nothing?
print(df.head(), "\n", df.shape)

2015 (2187, 38)
2016 (2186, 38)
2017 (2157, 38)
   area_fips  own_code industry_code  agglvl_code  size_code  year qtr  \
0      15000         0            10           50          0  2015   A   
1      15000         1            10           51          0  2015   A   
2      15000         1           101           52          0  2015   A   
3      15000         1          1013           53          0  2015   A   
4      15000         1           102           52          0  2015   A   

  disclosure_code  annual_avg_estabs  annual_avg_emplvl  ...  \
0             NaN              39077             637854  ...   
1             NaN                377              33041  ...   
2             NaN                  1               4771  ...   
3             NaN                  1               4771  ...   
4             NaN                376              28270  ...   

   oty_total_annual_wages_chg  oty_total_annual_wages_pct_chg  \
0                  1619241159                            

In [8]:
def toStringFips(fips):

    #if not equal to five, then we have to fix it by padding 0s on the left hand side...
    if len(str(fips)) != 5:
        fips = '{:05d}'.format(fips)
        print(fips)
    return str(fips)

def toFipsSeries(series):
    return series.apply(toStringFips)

county_list = toFipsSeries(county_fips["county_fips"])
print(type(county_list))

06037
04013
06073
06059
06065
06071
06085
06001
06067
06013
04019
06019
09110
06029
06111
06075
06077
08041
06081
08031
01073
08005
09190
08059
09170
06099
08001
06097
06107
04021
09140
06095
06083
06053
06061
01097
01089
05119
08035
08069
08123
09120
08013
05007
06047
02020
06079
09180
06087
06041
05143
01003
04025
01125
01117
01101
06113
04015
04027
06007
06017
06089
01081
06025
09130
08101
06039
08077
06031
04005
06055
06023
05131
05045
05125
04003
01103
01015
05031
09160
02170
01083
04017
01069
01055
06057
05051
01095
06101
09150
01077
02090
01115
01043
06045
01051
06115
01121
05145
08014
05085
01049
06033
06069
04001
06103
01127
05069
05115
08045
05033
02122
01001
01009
01113
01033
08067
08037
01031
06109
04007
01071
08043
01045
04023
05035
05055
06009
01021
08085
06093
05005
05091
06005
01123
04009
05093
05063
05009
05139
01039
01047
01053
01017
01079
05059
01109
01059
02110
08029
08117
06035
08087
01093
05015
06021
08039
06015
01061
08083
05071
01019
05023
08107
08119
01005
0113

# Final Assembly & Filtering

Here is the hardest part, I now need to iteratively read through all 3k counties and collect the data that I need from each one. This will take a while (understandably so), and so I need to implement checkpoints so I don't lose everything in one go.

I need some checkpoint method to do this... there is going to be quite some data. So far, I am retrieving 1 per county per year, so 3k counties * 24 years * 1 row = 72k rows in the final csv. That is quite a lot, and I am compelled to use a SQL database to make it easier for me to fetch things.

In [ ]:
# method to finally retrieve everything. i mean it works but how do we checkpoint
df = pd.DataFrame()
years = range(2018,2020,1)
for county in county_list:
    print("County: ", county)
    result = qcew_series(years, "area", county) #breaks at 09110
    # print("Size of dataframe: ", result.shape)
    result = result[result["industry_code"] == "238211"] # residential electricians
    # print("Observations kept: ", result.shape) #should do additional check if it actually is all electricians
    df = pd.concat([result, df], ignore_index= True)
    # print("Final dataframe shape: ", df.shape)

County:  06037
County:  17031
County:  48201
County:  04013
County:  06073
County:  06059
County:  12086
County:  36047
County:  48113
County:  06065
County:  32003
County:  36081
County:  53033
County:  06071
County:  48439
County:  48029
County:  12011
County:  06085
County:  26163
County:  06001
County:  25017
County:  36061
County:  06067
County:  42101
County:  12099
County:  36103
County:  12057
County:  12095
County:  36005
County:  36059
County:  39049
County:  48453
County:  26125
County:  27053
County:  39035
County:  42003
County:  49035
County:  37183
County:  06013
County:  48085
County:  37119
County:  51059
County:  13121
County:  24031
County:  04019
County:  12031
County:  06019
County:  15003
County:  36119
County:  29189
County:  13135
County:  48121
County:  09110


HTTPError: 404 Client Error: Not Found for url: https://data.bls.gov/cew/data/api/2018/a/area/09110.csv

In [12]:

url = "https://download.bls.gov/pub/time.series/ap/ap.data.0.Current"
r = requests.head(url)
if 'accept-ranges' in r.headers:
    print("Server supports resumable download")
else:
    print("Server does not support resumable download")

Server does not support resumable download
